# Kubernetes Failure Prediction: Model Training

This notebook focuses on training various machine learning models for predicting failures in Kubernetes clusters.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import time
import joblib

# Set up paths to access parent directory modules
import sys
sys.path.append('..')

# Import custom modules
from data_processor import apply_feature_engineering, handle_class_imbalance, apply_scaling
from model_trainer import train_random_forest, train_isolation_forest, train_time_series_model
from model_evaluator import get_feature_importance
from visualizer import plot_feature_importance
from utils import save_model

## Load Preprocessed Data

First, we'll load the preprocessed data that we created in the previous notebook.

In [ ]:
# Load preprocessed data
try:
    data = pd.read_csv('../data/preprocessed_kubernetes_data.csv')
    print(f"Loaded preprocessed data with shape: {data.shape}")
except FileNotFoundError:
    print("Preprocessed data not found. Please run the data_exploration notebook first.")
    # If you're running this directly, you can generate synthetic data here
    from data_generator import generate_kubernetes_data
    from data_processor import preprocess_data
    raw_data = generate_kubernetes_data(5000, 0.1, 30)
    data, _ = preprocess_data(raw_data)
    print(f"Generated and preprocessed synthetic data with shape: {data.shape}")

data.head()

## Feature Engineering

We'll apply feature engineering to enhance our dataset with temporal features and other derived metrics.

In [ ]:
# Define feature engineering parameters
window_sizes = [5, 10]  # Rolling window sizes for temporal features
use_pca = True
pca_components = 10

# Apply feature engineering
print("Applying feature engineering...")
feature_engineered_data = apply_feature_engineering(
    data, 
    window_sizes=window_sizes,
    apply_pca=use_pca,
    n_components=pca_components
)

print(f"Shape after feature engineering: {feature_engineered_data.shape}")
feature_engineered_data.head()

## Train-Test Split

We'll split our data into training and testing sets.

In [ ]:
# Separate features and target
X = feature_engineered_data.drop('failure', axis=1)
y = feature_engineered_data['failure']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Check class distribution in train and test sets
train_failure_rate = y_train.mean() * 100
test_failure_rate = y_test.mean() * 100
print(f"Failure rate in training set: {train_failure_rate:.2f}%")
print(f"Failure rate in testing set: {test_failure_rate:.2f}%")

## Handle Class Imbalance

If our dataset has an imbalanced class distribution, we'll handle it using techniques like SMOTE.

In [ ]:
# Apply class imbalance handling with SMOTE
imbalance_method = "SMOTE"  # Options: 'SMOTE', 'RandomUnderSampler', 'ADASYN'

if train_failure_rate < 20:  # If we have significant imbalance
    print(f"Applying {imbalance_method} to handle class imbalance...")
    X_train_balanced, y_train_balanced = handle_class_imbalance(X_train, y_train, method=imbalance_method)
    print(f"Shape after imbalance handling: {X_train_balanced.shape}")
    balanced_failure_rate = y_train_balanced.mean() * 100
    print(f"New failure rate in training set: {balanced_failure_rate:.2f}%")
else:
    print("Class imbalance is not significant. Skipping imbalance handling.")
    X_train_balanced, y_train_balanced = X_train, y_train

## Apply Scaling

We'll scale our features to ensure that all features contribute equally to the models.

In [ ]:
# Apply scaling
scaling_method = "RobustScaler"  # Better for data with outliers
print(f"Applying {scaling_method}...")

X_train_scaled, X_test_scaled, scaler = apply_scaling(
    X_train_balanced, X_test, method=scaling_method
)

# Save the scaler for future use
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler saved to ../models/scaler.pkl")

## Train Random Forest Model

Now we'll train a Random Forest model for classification-based prediction.

In [ ]:
# Train Random Forest model
rf_n_estimators = 100
rf_max_depth = 10

print(f"Training Random Forest model with {rf_n_estimators} estimators and max depth {rf_max_depth}...")
start_time = time.time()

rf_model = train_random_forest(
    X_train_scaled, 
    y_train_balanced,
    n_estimators=rf_n_estimators,
    max_depth=rf_max_depth
)

training_time = time.time() - start_time
print(f"Random Forest model trained in {training_time:.2f} seconds")

# Save the model
save_model(rf_model, '../models/random_forest_model.pkl')
print("Random Forest model saved to ../models/random_forest_model.pkl")

## Analyze Feature Importance

Let's examine which features are most important for predicting failures.

In [ ]:
# Get feature importance
feature_importance = get_feature_importance(
    rf_model, 
    list(X_train_scaled.columns),
    top_n=15  # Show top 15 features
)

# Plot feature importance
fig = plot_feature_importance(feature_importance, top_n=15)
plt.tight_layout()
plt.show()

# Save feature importance information
feature_importance.to_csv('../models/feature_importance.csv', index=False)
print("Feature importance saved to ../models/feature_importance.csv")

## Train Isolation Forest for Anomaly Detection

Next, we'll train an Isolation Forest model for anomaly detection.

In [ ]:
# Calculate default contamination value (proportion of failures in training set)
default_contamination = float(y_train_balanced.mean())
if_contamination = default_contamination
if_n_estimators = 100

print(f"Training Isolation Forest with contamination {if_contamination:.3f} and {if_n_estimators} estimators...")
start_time = time.time()

if_model = train_isolation_forest(
    X_train_scaled,
    contamination=if_contamination,
    n_estimators=if_n_estimators
)

training_time = time.time() - start_time
print(f"Isolation Forest model trained in {training_time:.2f} seconds")

# Save the model
save_model(if_model, '../models/isolation_forest_model.pkl')
print("Isolation Forest model saved to ../models/isolation_forest_model.pkl")

## Train Time Series Model (ARIMA)

Finally, we'll train a time series model for forecasting future metric values.

In [ ]:
# Check if our original dataset has a timestamp column (needed for time series analysis)
if 'timestamp' in data.columns:
    print("Training ARIMA time series model...")
    
    # Select a key metric for time series prediction
    ts_feature = "cpu_usage_percent"  # This could be changed to other metrics
    
    # Make sure we have the original, non-engineered data for time series modeling
    try:
        raw_data = pd.read_csv('../data/preprocessed_kubernetes_data.csv')
    except FileNotFoundError:
        from data_generator import generate_kubernetes_data
        raw_data = generate_kubernetes_data(5000, 0.1, 30)
    
    # Train the time series model
    start_time = time.time()
    ts_model = train_time_series_model(
        raw_data,
        feature=ts_feature
    )
    
    training_time = time.time() - start_time
    print(f"Time Series model trained in {training_time:.2f} seconds")
    
    # Save the time series model
    save_model(ts_model, '../models/time_series_model.pkl')
    print("Time Series model saved to ../models/time_series_model.pkl")
else:
    print("No timestamp column found in the data. Skipping time series modeling.")

## Save Prepared Training and Testing Data

Save the prepared data for later use in model evaluation.

In [ ]:
# Save training and testing data
train_test_data = {
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train_balanced,
    'y_test': y_test
}

joblib.dump(train_test_data, '../data/train_test_data.pkl')
print("Training and testing data saved to ../data/train_test_data.pkl")

## Conclusion

In this notebook, we've trained multiple models for Kubernetes failure prediction:

1. **Random Forest**: Classification-based prediction with feature importance analysis
2. **Isolation Forest**: Anomaly detection for identifying unusual patterns
3. **ARIMA**: Time series forecasting for predicting future metric values

The models have been saved and can be used for evaluation and prediction in the subsequent notebooks.